# 01 — Significance tests

**Question:** Is the conversion rate significantly higher during Black Friday–Cyber Monday than on normal days?

**Data:** BigQuery query `sql/08_black_friday_vs_normal.sql` on the GA4 obfuscated sample (Google Merchandise Store). Unit: distinct users who viewed at least one product. Normal days: Nov 1–20, 2020 (14,041 viewers, 866 buyers). Black Friday–Cyber Monday: Nov 27–30, 2020 (3,225 viewers, 333 buyers).

**Method:** Two-sided two-proportion z-test with a pooled rate, α = 5% (fixed before looking at the result). Computed by hand with NumPy, then checked with `statsmodels`. We also report the 95% confidence interval of the difference.

In [3]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest
print("ok")

ok


In [7]:
# Data (query 08)
x_normal, n_normal = 866, 14041   # buyers, viewers
x_bf, n_bf = 333, 3225

# Step 2: conversion rate of each group
p_normal = x_normal / n_normal
p_bf = x_bf / n_bf

# Step 3: pooled rate under H0
p_pool = (x_normal + x_bf) / (n_normal + n_bf)

# Step 4: standard error under H0, then z
se = np.sqrt(p_pool * (1-p_pool) * (1 / n_normal + 1 / n_bf))
z = (p_bf - p_normal) / se

# Step 5: two-sided p-value
p_value = 2 * stats.norm.sf(abs(z))

print(f"p_normal = {p_normal:.4f}, p_bf = {p_bf:.4f}")
print(f"z = {z:.2f}, p-value = {p_value:.2e}")

p_normal = 0.0617, p_bf = 0.1033
z = 8.38, p-value = 5.45e-17


In [9]:
from statsmodels.stats.proportion import confint_proportions_2indep

# Check with statsmodels (BF first so that z > 0)
z_sm, p_sm = proportions_ztest(count=[333, 866], nobs=[3225, 14041])
print(f"statsmodels: z = {z_sm:.2f}, p-value = {p_sm:.2e}")

# 95% CI of the difference p_bf - p_normal
low, high = confint_proportions_2indep(333, 3225, 866, 14041, method="wald")
print(f"Difference = {100*(p_bf - p_normal):.1f} pts, 95% CI = [{100*low:.1f} ; {100*high:.1f}] pts")
print(f"Relative lift = {100*(p_bf/p_normal - 1):.0f}%")

statsmodels: z = 8.38, p-value = 5.45e-17
Difference = 4.2 pts, 95% CI = [3.0 ; 5.3] pts
Relative lift = 67%


## Conclusion

Conversion is significantly higher during Black Friday–Cyber Monday than on normal days: 10.3% vs 6.2% (z = 8.38, p < 0.001), so the gap is very unlikely to be due to chance.

The effect is large: +4.2 percentage points (95% CI: +3.0 to +5.3 pts), i.e. a +67% relative lift.

**Limits:** this is an observational comparison, not a randomized experiment, so it shows *that* conversion was higher, not *why* (discounts, ads, holiday shoppers already decided to buy). The two periods also have different lengths (20 vs 4 days), and the same user can appear in both groups, which breaks the independence assumption of the test.

In [11]:
df = pd.read_csv("../data/04_funnel_by_country.csv")
print(df.shape)
df

(10, 7)


,country,views,add_to_carts,checkouts,payment_infos,purchases,conversion_rate
0,United States,26953,5603,4310,2516,1942,7.2
1,India,5795,1162,878,524,406,7.0
2,Canada,4653,993,764,465,355,7.6
3,United Kingdom,1941,398,297,170,137,7.1
4,France,1205,234,187,119,93,7.7
5,Spain,1119,232,196,108,84,7.5
6,China,1049,224,157,79,68,6.5
7,Taiwan,1033,208,150,98,72,7.0
8,Germany,1019,193,166,102,74,7.3
9,Italy,878,164,126,61,50,5.7


In [12]:
# Conversion rate and 95% CI for each country (one proportion)
df["p"] = df["purchases"] / df["views"]
df["se"] = np.sqrt(df["p"] * (1 - df["p"]) / df["views"])
df["ci_low"] = 100 * (df["p"] - 1.96 * df["se"])
df["ci_high"] = 100 * (df["p"] + 1.96 * df["se"])
df["ci_width"] = df["ci_high"] - df["ci_low"]

df[["country", "views", "purchases", "ci_low", "ci_high", "ci_width"]].round(1)

,country,views,purchases,ci_low,ci_high,ci_width
0,United States,26953,1942,6.9,7.5,0.6
1,India,5795,406,6.3,7.7,1.3
2,Canada,4653,355,6.9,8.4,1.5
3,United Kingdom,1941,137,5.9,8.2,2.3
4,France,1205,93,6.2,9.2,3.0
5,Spain,1119,84,6.0,9.1,3.1
6,China,1049,68,5.0,8.0,3.0
7,Taiwan,1033,72,5.4,8.5,3.1
8,Germany,1019,74,5.7,8.9,3.2
9,Italy,878,50,4.2,7.2,3.1


In [13]:
# Best vs worst country: France vs Italy
fr = df[df["country"] == "France"].iloc[0]
it = df[df["country"] == "Italy"].iloc[0]

z_fi, p_fi = proportions_ztest(count=[fr["purchases"], it["purchases"]],
                               nobs=[fr["views"], it["views"]])

# Bonferroni: we implicitly compared all pairs of the 10 countries
n_pairs = 10 * 9 // 2
alpha_bonf = 0.05 / n_pairs

print(f"France {100*fr['p']:.1f}% vs Italy {100*it['p']:.1f}%")
print(f"z = {z_fi:.2f}, p-value = {p_fi:.3f}, Bonferroni alpha = {alpha_bonf:.4f}")

France 7.7% vs Italy 5.7%
z = 1.80, p-value = 0.071, Bonferroni alpha = 0.0011


## Conversion by country

The 95% confidence intervals show how much the country rates can be trusted: ±0.3 pt for the United States (26,953 viewers) but about ±1.5 pt for the smaller markets (around 1,000 viewers each).

The widest gap in the top 10, France (7.7%) vs Italy (5.7%), is **not statistically significant**: z = 1.80, p = 0.07 > 0.05. Since this pair was picked *after* looking at the data, the right comparison is with a Bonferroni threshold over the 45 possible pairs (α = 0.05 / 45 ≈ 0.001), which it misses by far.

**Takeaway:** conversion is essentially the same across the top 10 countries. Country is not a lever worth prioritising; the checkout step is.